# Notebook 17 (Revised) — PVOD cross-dataset validation preflight

**Status:** New, not yet executed on the local PVOD dataset. **Read-only data/feature/protocol audit; no training or scoring.**

**Research question:** Do the LightGBM hyperparameters and feature-family choices selected in the revised SOLETE development study retain forecasting utility when independently *refitted within each PVOD station*? This is **transfer of a preselected specification, not transfer of SOLETE fitted weights**. Models will be **F1 (15 min)** and **F2 (30/60 min)**.

**Scientific classification:** The historical PVOD research notebook has already examined PVOD station data and reported test results. Therefore this revised study is *cross-dataset replication on previously studied external data*, **not a previously unseen/blind test**. The earlier F3-vs-F4 PVOD experiments answer a distinct representation question and are never used to reselect the revised primary models.

**What this notebook does:** inspect all ten local PVOD station files, reuse the tested cleaning/gap-safe sample builders, reconstruct fixed station chronological splits, establish **availability-safe train+validation refit cutoffs**, and audit the selected training design matrices. It may read complete station measurements to reconstruct history and sample geometry (including stored test targets inside the frozen source sample builder), but it **never reports test performance, fits models, predicts, or writes files**.

**Next step, separate authorization:** Once this preflight passes and is reviewed, freeze the evaluation implementation before any 30 station-specific LightGBM refits and test scoring. Do not alter the completed SOLETE notebooks or the original PVOD notebook.

## 1. Frozen cross-dataset specification

| Horizon | SOLETE-selected architecture | PVOD representation | PVOD predictors | Allowed hyperparameters |
|---|---|---|---:|---|
| 15 min | LightGBM C1 | F1 = PVOD F0 + target-time Gregorian | 86 | 300 trees, 0.03 learning rate, 15 leaves, seed 42 |
| 30 min | LightGBM C1 | F2 = PVOD F1 + site-specific target-time solar | 91 | Same |
| 60 min | LightGBM C1 | F2 | 91 | Same |

PVOD target = **observed power / station nominal capacity**, measured in **per unit (p.u.)**, without clipping values greater than 1. Station-specific temporal splits use the first 65% / next 15% / final 20% of each station's *time span*. Their labels depend on **target-valid time**; actual eligible forecasting rows must respect missing 15-minute intervals. Gregorian and solar features are deterministic at target-valid time. The PVOD F0 channels (including NWP and LMD weather fields) are indexed at forecast issue time or earlier, not at future target-valid time. **CSV timestamps alone do not establish the real-world publication/issuance times of NWP forecasts**, so operational forecast availability remains a limitation rather than a proved guarantee.

For deployment-safe final station refits, only train/validation labels that arrive **strictly before the earliest eligible test forecast issue across the three horizons** are allowed. This supplements the historical PVOD 65/15/20 protocol with an explicit label-availability guard and changes its near-boundary training populations. We will disclose this protocol difference rather than quietly equating old and revised PVOD results.

In [1]:
from __future__ import annotations
from pathlib import Path
import hashlib
import json
import os
import sys
import numpy as np
import pandas as pd
from IPython.display import display

# There is intentionally no authorization option that can cause fitting/scoring.
TRAINING_AUTHORIZED = False
PVOD_SCORING_AUTHORIZED = False
WRITE_ARTIFACTS = False
assert not TRAINING_AUTHORIZED and not PVOD_SCORING_AUTHORIZED and not WRITE_ARTIFACTS

candidates = []
if os.environ.get("SOLETE_PROJECT_ROOT"):
    candidates.append(Path(os.environ["SOLETE_PROJECT_ROOT"]).expanduser())
candidates.extend((Path.cwd(), *Path.cwd().parents))
PROJECT_ROOT = next((p.resolve() for p in candidates
    if (p / ".git").exists()
    and (p / "src/pvod.py").is_file()
    and (p / "results/development_only/revised_model_selection_lock_v1.json").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open from the SOLETE project (or set SOLETE_PROJECT_ROOT)")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
PVOD_ROOT = PROJECT_ROOT / "external_data" / "PVOD"
if not PVOD_ROOT.is_dir():
    raise FileNotFoundError(f"PVOD files required at {PVOD_ROOT}. Keep them outside Git.")
print("PVOD preflight: read-only mode")
print("PVOD source directory:", PVOD_ROOT)
print("SOLETE predictions/fits: not loaded or run")

PVOD preflight: read-only mode
PVOD source directory: /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/external_data/PVOD
SOLETE predictions/fits: not loaded or run


## 2. Read the model-selection specification, not fitted SOLETE model weights

Only the configuration/feature-family choices are transferred. No SOLETE `.txt` weight file is opened. Check the exact frozen model factory parameters and the existing PVOD frozen LightGBM defaults. This preflight does **not** make model fits.

In [2]:
from src.classical_models import LIGHTGBM_CONFIGS, LIGHTGBM_FIXED_PARAMS
from src.pvod import (
    PVOD_CADENCE, PVOD_EXPECTED_DIMS, PVOD_HORIZONS,
    PVOD_LIGHTGBM_BASE_CONFIG, PVOD_TIMESTAMP_CORRECTIONS,
    PVOD_METEOROLOGICAL_COLUMNS,
    build_pvod_site_registry, clean_pvod_station,
    build_pvod_split_boundaries, build_pvod_f0,
    build_pvod_solar_features, build_pvod_sample_indices,
    materialize_pvod_design,
)
from src.preprocessing import build_gregorian_features

lock_path = PROJECT_ROOT / "results/development_only/revised_model_selection_lock_v1.json"
lock = json.loads(lock_path.read_text(encoding="utf-8"))
assert lock["evaluation_policy"]["terminal_access_authorized"] is False
locked = {int(z["horizon_minutes"]): z for z in lock["selected_candidates"]}
assert set(locked) == set(PVOD_HORIZONS) == {15, 30, 60}
FAMILIES = {h: locked[h]["feature_family"] for h in (15, 30, 60)}
assert FAMILIES == {15: "F1", 30: "F2", 60: "F2"}
CONFIG = {"n_estimators": 300, "learning_rate": 0.03, "num_leaves": 15}
assert dict(LIGHTGBM_CONFIGS[0]) == CONFIG
assert all(z["model"] == "LightGBM" and z["candidate_id"] == "LightGBM_C1"
           and z["configuration"] == CONFIG for z in locked.values())
assert {**LIGHTGBM_FIXED_PARAMS, **CONFIG} == PVOD_LIGHTGBM_BASE_CONFIG
assert PVOD_CADENCE == pd.Timedelta(minutes=15)
assert PVOD_EXPECTED_DIMS["F1"] == 86 and PVOD_EXPECTED_DIMS["F2"] == 91
assert set(PVOD_TIMESTAMP_CORRECTIONS) == {"station00"}
print("Frozen LightGBM specification: PASSED")
print("PVOD family transfer:", FAMILIES)
print("Parameter transfer only: SOLETE fitted weights are not used")

Frozen LightGBM specification: PASSED
PVOD family transfer: {15: 'F1', 30: 'F2', 60: 'F2'}
Parameter transfer only: SOLETE fitted weights are not used


## 3. PVOD release inventory and capacity metadata

Expect `metadata.csv` and `station00.csv` through `station09.csv`. The supplementary McClear files and ephemeris are **not required** for the selected F1/F2 models. Site coordinates/capacities come from metadata. No outputs are exported.

In [3]:
expected_stations = tuple(f"station{i:02d}" for i in range(10))
station_paths = {name: PVOD_ROOT / f"{name}.csv" for name in expected_stations}
missing = [str(p) for p in [PVOD_ROOT / "metadata.csv", *station_paths.values()]
           if not p.is_file()]
assert not missing, f"Missing PVOD release files: {missing}"
assert {p.stem for p in PVOD_ROOT.glob("station*.csv")} == set(expected_stations), (
    "Expected exactly ten core station CSVs")
metadata = pd.read_csv(PVOD_ROOT / "metadata.csv")
registry = build_pvod_site_registry(metadata)
assert len(registry) == 10
assert set(registry["station"]) == set(expected_stations)
assert registry["capacity_MW"].gt(0).all()
site_lookup = registry.set_index("station").loc[list(expected_stations)]
display(site_lookup[["capacity_MW", "latitude_deg", "longitude_deg"]])
print("PVOD RELEASE + 10-STATION METADATA: PASSED")

,capacity_MW,latitude_deg,longitude_deg
station,,,
station00,6.6,38.047780,114.951390
station01,20.0,38.183060,117.457220
station02,17.0,38.057280,114.198870
station03,20.0,38.109560,114.114940
station04,20.0,39.515500,114.867670
station05,35.0,38.235500,114.123600
station06,15.0,36.898910,114.548410
station07,20.0,36.644030,113.641870
station08,20.0,36.707610,113.899990


PVOD RELEASE + 10-STATION METADATA: PASSED


## 4. Predeclare population and evaluation policies before building designs

**Population:** the existing `src.pvod` exact-time/gap-safe horizon samples. A station test row is defined by its **target-valid** timestamp in the last 20% of its chronological time span. Test samples are not silently removed due to disagreement with model predictions; they are fixed by source-data availability alone.

**Station refit (future step):** train a *new* LightGBM model per station × horizon on all eligible train+validation samples whose target labels are available **before the earliest test forecast issue**. No tuning, validation early stopping, feature family substitutions, SOLETE weight reuse, or test-dependent stopping. The chosen strict cutoff is calculated from timestamp/sample geometry, not performance outcomes.

**Future primary metrics:** station-wise MAE and RMSE in capacity-normalized p.u.; macro station mean (equal weight to all ten sites), with per-station metrics shown. Report pooled observation-weighted metrics as secondary and MW metrics as site-specific supplemental information. For daylight analysis use solar elevation > 0 at **target-valid** time.

**Baselines:** issue-time PV persistence (available for every eligible test issue); previous-day target-valid-time PV persistence (may be unavailable across gaps). For comparisons involving the previous-day baseline, report its **explicit matched subset and count**, in addition to the full gap-safe test population. Never interpolate missing days or quietly drop samples. No significance claims based merely on ten sites.

**Interpretation:** this is a revised replication on the previously studied PVOD release, not a fully untouched external test. Separate historical F3/F4 claims remain outside this primary F1/F2 experiment.

In [4]:
# Use source functions for the fixed 65/15/20 chronological split.
# Confirm that our embargo policy is independent of held-out TARGET VALUES:
# the cutoff depends only on gap-safe issue/valid-time geometry.
def station_cutoff_and_masks(samples_by_horizon):
    first_issues = {}
    for h in (15, 30, 60):
        sample = samples_by_horizon[h]
        assert {"train", "validation", "test"}.issubset(set(sample["split"]))
        test = sample.loc[sample["split"].eq("test")]
        assert not test.empty
        first_issues[h] = pd.Timestamp(test["issue_time_utc"].iloc[0])
    cutoff = min(first_issues.values())
    masks = {}
    for h in (15, 30, 60):
        sample = samples_by_horizon[h]
        trainval = sample["split"].isin(["train", "validation"])
        # Availability-safe: every training target is known at ALL first test issues.
        safe = trainval & sample["target_time_utc"].lt(cutoff)
        assert safe.any() and safe.sum() <= trainval.sum()
        assert sample.loc[safe, "target_time_utc"].max() < first_issues[h]
        assert not (safe & sample["split"].eq("test")).any()
        masks[h] = safe
    return cutoff, first_issues, masks

print("PVOD station refit cutoff: minimum first eligible test issue across 15/30/60 min")
print("Label availability rule: training target-valid time < refit cutoff (strict)")

PVOD station refit cutoff: minimum first eligible test issue across 15/30/60 min
Label availability rule: training target-valid time < refit cutoff (strict)


## 5. Real ten-station split, gap and F1/F2 input preflight

This step **reads the ten PVOD CSVs** and uses the existing station cleaning, including the frozen guarded timestamp correction for `station00`. It preserves genuine gaps and above-capacity measurements; it computes deterministic solar position at the **exact PVOD valid time** (no SOLETE +2-minute convention). It creates only the **future refit training** design for input audit; **no test predictors are scored** and **no model is fitted**.

The frozen PVOD sample builder holds test target values in its sample-table structures while checking data eligibility. This is **not a blind-data preflight** and is not represented as one. The code does not calculate, display or optimize using PVOD test outcome metrics.

In [5]:
from collections import defaultdict
import gc

summary_rows = []
schema_hashes = defaultdict(set)
for station in expected_stations:
    site = site_lookup.loc[station]
    raw = pd.read_csv(station_paths[station])
    required = {"date_time", "power", *PVOD_METEOROLOGICAL_COLUMNS}
    assert required.issubset(raw.columns), (station, sorted(required-set(raw.columns)))
    assert raw[list(PVOD_METEOROLOGICAL_COLUMNS) + ["power"]].notna().all().all(), (
        station, "unexpected raw data missing values")
    clean = clean_pvod_station(raw, station_id=station, capacity_kw=float(site["capacity_kW"]))
    assert clean["timestamp_utc"].is_unique and clean["timestamp_utc"].is_monotonic_increasing
    assert not clean["power_pu"].isna().any()
    f0 = build_pvod_f0(clean)
    assert f0.shape == (len(clean), 71)
    times = pd.DatetimeIndex(clean["timestamp_utc"])
    f1 = build_gregorian_features(times)
    f2 = build_pvod_solar_features(times, latitude_deg=float(site["latitude_deg"]),
                                    longitude_deg=float(site["longitude_deg"]))
    assert f1.shape == (len(clean), 15) and f2.shape == (len(clean), 5)
    boundary = build_pvod_split_boundaries({station: clean}).iloc[0]
    samples = build_pvod_sample_indices(clean, f0, boundary, f2)
    cutoff, first_issues, safe_masks = station_cutoff_and_masks(samples)

    for h in (15, 30, 60):
        s = samples[h]
        family = FAMILIES[h]
        refit_s = s.loc[safe_masks[h]].copy()
        test_s = s.loc[s["split"].eq("test")].copy()
        assert not refit_s.empty and not test_s.empty
        assert (refit_s["target_time_utc"] < cutoff).all()
        assert (test_s["issue_time_utc"] >= first_issues[h]).all()
        assert ((s["target_time_utc"] - s["issue_time_utc"])
                == pd.Timedelta(minutes=h)).all()
        # Materialize only safe TRAIN+VALIDATION samples, not test designs.
        design = materialize_pvod_design(
            refit_s, f0=f0, f1=f1, f2=f2,
            f3=pd.DataFrame(index=times), f4=pd.DataFrame(index=times),
            feature_block=family,
        )
        X = design["X"]
        assert X.shape == (len(refit_s), PVOD_EXPECTED_DIMS[family])
        assert X.index.equals(design["y_pu"].index)
        assert np.isfinite(X.to_numpy(dtype=float)).all()
        assert np.isfinite(design["y_pu"].to_numpy(dtype=float)).all()
        schema = hashlib.sha256(json.dumps(list(X.columns), ensure_ascii=False).encode()).hexdigest()
        schema_hashes[family].add(schema)
        # Prior-day target-time baseline availability is a timestamp-only question.
        yday_times = pd.DatetimeIndex(test_s["target_time_utc"]) - pd.Timedelta(days=1)
        has_yday = yday_times.isin(times)
        assert len(has_yday) == len(test_s)
        summary_rows.append({
            "station": station, "horizon_min": h, "family": family,
            "observations_total": len(clean),
            "train_rows_original": int(s["split"].eq("train").sum()),
            "validation_rows_original": int(s["split"].eq("validation").sum()),
            "refit_rows_safe": len(refit_s),
            "purged_near_test": int(s["split"].isin(["train", "validation"]).sum() - len(refit_s)),
            "test_rows_gap_safe": len(test_s),
            "test_rows_yesterday_available": int(has_yday.sum()),
            "test_rows_yesterday_unavailable": int((~has_yday).sum()),
            "refit_cutoff_utc": cutoff.isoformat(),
            "first_test_issue_utc": first_issues[h].isoformat(),
            "test_start_valid_utc": pd.Timestamp(test_s["target_time_utc"].iloc[0]).isoformat(),
            "feature_columns": X.shape[1], "training_schema_sha256": schema,
        })
        del design, X, refit_s, test_s
    print(f"{station}: data/splits/refit-label cutoff/three feature schemas PASSED")
    del raw, clean, f0, f1, f2, samples
    gc.collect()

pvod_preflight = pd.DataFrame(summary_rows).sort_values(["station", "horizon_min"]).reset_index(drop=True)
assert len(pvod_preflight) == 30
assert pvod_preflight.groupby("station")["horizon_min"].nunique().eq(3).all()
assert (pvod_preflight["refit_rows_safe"] > 0).all()
assert (pvod_preflight["test_rows_gap_safe"] > 0).all()
assert (pvod_preflight["purged_near_test"] >= 0).all()
assert {family:len(hashes) for family,hashes in schema_hashes.items()} == {"F1":1, "F2":1}
display(pvod_preflight[["station","horizon_min","family","refit_rows_safe",
    "purged_near_test", "test_rows_gap_safe", "test_rows_yesterday_unavailable"]])
print("PVOD 10-STATION GAP-SAFE INPUT PREFLIGHT: PASSED — 30 station-horizon designs")

station00: data/splits/refit-label cutoff/three feature schemas PASSED
station01: data/splits/refit-label cutoff/three feature schemas PASSED
station02: data/splits/refit-label cutoff/three feature schemas PASSED
station03: data/splits/refit-label cutoff/three feature schemas PASSED
station04: data/splits/refit-label cutoff/three feature schemas PASSED
station05: data/splits/refit-label cutoff/three feature schemas PASSED
station06: data/splits/refit-label cutoff/three feature schemas PASSED
station07: data/splits/refit-label cutoff/three feature schemas PASSED
station08: data/splits/refit-label cutoff/three feature schemas PASSED
station09: data/splits/refit-label cutoff/three feature schemas PASSED


,station,horizon_min,family,refit_rows_safe,purged_near_test,test_rows_gap_safe,test_rows_yesterday_unavailable
0,station00,15,F1,22900,4,5798,0
1,station00,30,F2,22898,4,5798,0
2,station00,60,F2,22894,4,5798,0
3,station01,15,F1,26626,4,6681,0
4,station01,30,F2,26625,4,6681,0
5,station01,60,F2,26623,4,6681,0
6,station02,15,F1,24321,4,5719,0
7,station02,30,F2,24319,4,5717,0
8,station02,60,F2,24315,4,5713,0
9,station03,15,F1,11650,4,2937,0


PVOD 10-STATION GAP-SAFE INPUT PREFLIGHT: PASSED — 30 station-horizon designs


## 6. Summary and no-fit/no-score acceptance gate

All counts are **input-population counts**, not performance estimates. The next evaluation must use precisely these fixed sample populations, label-availability masks, model features, and station boundaries. The model-vs-issue-persistence comparison uses all eligible test rows; the three-way comparison with previous-day persistence uses the **explicit matched subset** reported here. No PVOD hyperparameter or feature selection is allowed from the future scores. As PVOD was previously studied, label outcomes **cross-dataset retrospective replication**.

In [6]:
assert not TRAINING_AUTHORIZED and not PVOD_SCORING_AUTHORIZED and not WRITE_ARTIFACTS
assert len(pvod_preflight) == 10*3
assert pvod_preflight.groupby("horizon_min")["station"].nunique().eq(10).all()
assert pvod_preflight.groupby("station")["refit_cutoff_utc"].nunique().eq(1).all()
assert all(int(row.feature_columns) == PVOD_EXPECTED_DIMS[row.family]
           for row in pvod_preflight.itertuples(index=False))
print("PVOD CROSS-DATASET PREFLIGHT: PASSED")
print("Ten stations / three horizons / F1–F2 source-aligned training matrices: PASSED")
print("One label-availability-safe cutoff per station: PASSED")
print("Existing PVOD F3/F4 performance results: NOT READ")
print("SOLETE model weights: NOT LOADED; NO MODEL FITS OR PREDICTIONS")
print("PVOD test targets may have been loaded for sample eligibility, but no metrics are computed")
print("NO FILE WRITES | PVOD SCORING: DISABLED")
print("Further experimental stage: station-specific fitting + predeclared PVOD scoring")

PVOD CROSS-DATASET PREFLIGHT: PASSED
Ten stations / three horizons / F1–F2 source-aligned training matrices: PASSED
One label-availability-safe cutoff per station: PASSED
Existing PVOD F3/F4 performance results: NOT READ
SOLETE model weights: NOT LOADED; NO MODEL FITS OR PREDICTIONS
PVOD test targets may have been loaded for sample eligibility, but no metrics are computed
NO FILE WRITES | PVOD SCORING: DISABLED
Further experimental stage: station-specific fitting + predeclared PVOD scoring


## Interpretation and next action

This notebook provides an executable population/feature audit, **not** validation evidence of predictive skill. Review the displayed station counts and the size of any gap-related baseline exclusions before implementing station-specific training. **Do not replace or edit** the existing PVOD study and do not copy its historical F3/F4 results into this revised LightGBM F1/F2 primary comparison.

A later training/scoring notebook should be separately gated, freeze the 30 station–horizon fits **before generating any new PVOD metrics**, and report station-level p.u. MAE/RMSE against issue persistence and the explicitly matched previous-day baseline. Report equal-station macro averages and pooled summaries separately. Keep historical PVOD exposure visible in the paper. Do not overwrite existing model/result archives.